# 4.6 Obstacle Investigation & Labeling

Interactive map of all extracted clusters with inline labeling.

- **Click any dot** on the map to inspect the cluster and see its current label.
- Use **Accept**, **Update label**, or **Flag for review** to record your decision.
- All changes are saved to `inventory.csv` immediately.

**Dot appearance:** hollow = not yet reviewed · filled white edge = confirmed · red edge = flagged for review.

In [ ]:
import sys, io
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import numpy as np
import pandas as pd
import laspy
import ipywidgets as widgets
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

from config import CLUSTERS_DIR, LABELED_DIR

In [ ]:
INV_PATH = CLUSTERS_DIR / 'inventory.csv'
inv = pd.read_csv(INV_PATH)

# Ensure labeling columns exist
for col, default in [('final_label', None), ('label_source_final', None), ('needs_review', False)]:
    if col not in inv.columns:
        inv[col] = default

TILECODES = sorted(inv['tilecode'].unique().tolist())
print(f'Loaded {len(inv)} clusters across {len(TILECODES)} tile(s)')
print(inv.groupby('label')['cluster_idx'].count().rename('count').to_string())

In [ ]:
LABEL_NAMES = {
    0: 'Unknown', 1: 'Road', 9: 'Ground', 10: 'Building',
    30: 'Tree', 40: 'Car', 44: 'Bicycle', 50: 'Person',
    60: 'Street Light', 61: 'Traffic Light', 62: 'Traffic Sign',
    65: 'Bollard', 67: 'Stop Pole', 80: 'City Bench',
    81: 'Rubbish Bin', 83: 'Large Container', 85: 'Parking Meter',
    88: 'Bicycle Rack', 99: 'Noise / False Positive',
}

# Choices available in the label dropdown
LABEL_OPTIONS = [(f"{v}  ({k})", k) for k, v in sorted(LABEL_NAMES.items())]

DOT_COLORS  = {0:'#aaaaaa', 30:'#44ee44', 40:'#ff8800', 60:'#44aaff', 83:'#dd44dd'}
DOT_DEFAULT = '#ffffff'

LABEL_PRIORITY = {10:8, 1:7, 30:6, 40:5, 60:4, 83:4, 79:3, 90:3, 9:2, 0:1}
BG_RGB = {
    -1:(0.07,0.07,0.07),  0:(0.22,0.22,0.22),  1:(0.75,0.20,0.20),
     9:(0.50,0.50,0.50), 10:(0.20,0.35,0.70), 30:(0.20,0.65,0.20),
    40:(1.00,0.50,0.10), 60:(1.00,0.95,0.20), 79:(0.80,0.40,0.00),
    83:(0.70,0.20,0.70), 90:(0.90,0.60,0.10),
}
GRID_RES = 0.25

In [ ]:
# ── tile loading + 2-D background grid ────────────────────────────────────────
_tile_cache = {}
_bg_cache   = {}

def _load_tile_raw(tilecode):
    if tilecode in _tile_cache:
        return _tile_cache[tilecode]
    laz_path = LABELED_DIR / f'bgt_labeled_{tilecode}.laz'
    print(f'  Loading {laz_path.name}…', end=' ', flush=True)
    pc  = laspy.read(laz_path)
    xy  = np.column_stack([np.asarray(pc.x, np.float32),
                           np.asarray(pc.y, np.float32)])
    has = 'label' in pc.point_format.extra_dimension_names
    lbl = np.asarray(pc.label, np.int32) if has else np.zeros(len(xy), np.int32)
    print(f'{len(xy):,} pts')
    _tile_cache[tilecode] = (xy, lbl)
    return xy, lbl

def _make_bg(xy, labels):
    x, y   = xy[:,0], xy[:,1]
    x0, y0 = float(x.min()), float(y.min())
    xi = np.floor((x - x0) / GRID_RES).astype(np.int32)
    yi = np.floor((y - y0) / GRID_RES).astype(np.int32)
    nx, ny = int(xi.max())+1, int(yi.max())+1
    order  = np.argsort(np.vectorize(lambda l: LABEL_PRIORITY.get(int(l), 0))(labels))
    grid   = np.full((ny, nx), -1, np.int32)
    grid[yi[order], xi[order]] = labels[order]
    rgb    = np.full((ny, nx, 3), BG_RGB[-1], np.float32)
    for lv, c in BG_RGB.items():
        m = grid == lv
        if m.any(): rgb[m] = c
    return rgb, [x0, x0 + nx*GRID_RES, y0, y0 + ny*GRID_RES]

def _get_bg(tilecode):
    if tilecode not in _bg_cache:
        xy, lbl = _load_tile_raw(tilecode)
        _bg_cache[tilecode] = _make_bg(xy, lbl)
    return _bg_cache[tilecode]

In [ ]:
# ── detail renderer (RGB colours from scan) ────────────────────────────────────
def _sa(ax):
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
    ax.tick_params(labelsize=6, colors='grey')

def _fig_bytes(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, facecolor='#1a1a1a', bbox_inches='tight')
    buf.seek(0)
    return buf.read()

def render_detail(row):
    """Top-view (XY) + side-view (XZ) coloured by scan RGB."""
    try:
        npz = np.load(row['npz_path'])
    except Exception as e:
        fig = Figure(figsize=(9, 4), facecolor='#1a1a1a')
        FigureCanvasAgg(fig)
        ax = fig.add_subplot(111); _sa(ax)
        ax.text(0.5, 0.5, f'Cannot load NPZ:\n{e}', color='#cc4444',
                ha='center', va='center', transform=ax.transAxes)
        return _fig_bytes(fig)

    xyz    = npz['xyz_centered']
    colors = np.clip(npz['rgb_norm'], 0, 1)
    pt_sz  = max(1, min(10, 3000 // max(len(xyz), 1)))

    lbl   = int(row['label'])
    flbl  = row.get('final_label', None)
    name  = LABEL_NAMES.get(lbl, f'Label {lbl}')
    fname = LABEL_NAMES.get(int(flbl), f'Label {flbl}') if pd.notna(flbl) else 'not reviewed'
    title = (f"#{int(row['cluster_idx'])}  auto: {name} ({lbl})  ·  "
             f"confirmed: {fname}  ·  "
             f"{int(row['n_raw_pts']):,} pts  ·  {float(row['area_m2']):.2f} m²")

    fig = Figure(figsize=(10, 4.5), facecolor='#1a1a1a')
    FigureCanvasAgg(fig)
    fig.suptitle(title, color='white', fontsize=8)
    ax_t = fig.add_subplot(1, 2, 1)
    ax_s = fig.add_subplot(1, 2, 2)
    _sa(ax_t); _sa(ax_s)

    ax_t.scatter(xyz[:,0], xyz[:,1], c=colors, s=pt_sz, linewidths=0)
    ax_t.set_aspect('equal')
    ax_t.set_title('top view (XY)', color='#aaaaaa', fontsize=8)
    ax_t.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_t.set_ylabel('ΔY (m)', color='grey', fontsize=7)

    ax_s.scatter(xyz[:,0], xyz[:,2], c=colors, s=pt_sz, linewidths=0)
    ax_s.set_aspect('equal')
    ax_s.set_title('side view (XZ)', color='#aaaaaa', fontsize=8)
    ax_s.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_s.set_ylabel('height (m)', color='grey', fontsize=7)

    fig.tight_layout()
    return _fig_bytes(fig)

In [ ]:
# ── labeling helpers ───────────────────────────────────────────────────────────
def _save_inv():
    inv.to_csv(INV_PATH, index=False)

def _progress_str(tilecode):
    tile = inv[inv['tilecode'] == tilecode]
    reviewed = tile['final_label'].notna().sum()
    flagged  = tile['needs_review'].fillna(False).sum()
    return (f'<span style="color:#aaa;font-size:12px">'f'Tile: '
            f'<b>{reviewed}/{len(tile)}</b> reviewed'
            f'  ·  <span style="color:#ff6666">{flagged} flagged</span></span>')

def _dot_style(row):
    """Return (facecolor, edgecolor, edgewidth, alpha) for a cluster dot."""
    color = DOT_COLORS.get(int(row['label']), DOT_DEFAULT)
    if row.get('needs_review', False):
        return color, '#ff3333', 1.8, 1.0   # flagged: red edge
    if pd.notna(row.get('final_label', None)):
        return color, '#ffffff', 1.5, 1.0   # confirmed: white edge
    return color, '#111111', 0.5, 0.65      # unreviewed: dim, thin edge

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

# ── state ─────────────────────────────────────────────────────────────────────
_scatter_rows  = []          # [(scatter_artist, sub_df), …]
_selected_idx  = [None]      # inv.index of currently selected cluster
_current_tile  = [None]

# ── right-panel widgets ───────────────────────────────────────────────────────
info_html    = widgets.HTML(value='<span style="color:#666;font-size:12px">click a cluster dot</span>')
progress_html = widgets.HTML(value='')

label_dd = widgets.Dropdown(
    options=LABEL_OPTIONS, value=0,
    description='Label:',
    layout=widgets.Layout(width='280px'),
    style={'description_width': '50px'},
)
btn_accept = widgets.Button(description='✔ Accept', button_style='success',
                            layout=widgets.Layout(width='100px'),
                            tooltip='Confirm the current auto-label as-is')
btn_update = widgets.Button(description='✎ Update', button_style='primary',
                            layout=widgets.Layout(width='100px'),
                            tooltip='Save the label chosen in the dropdown')
btn_flag   = widgets.Button(description='⚑ Flag', button_style='warning',
                            layout=widgets.Layout(width='100px'),
                            tooltip='Mark as uncertain / needs review')

detail_img = widgets.Image(value=b'', format='png',
                           layout=widgets.Layout(width='560px'))

btn_row = widgets.HBox([label_dd, btn_accept, btn_update, btn_flag],
                       layout=widgets.Layout(gap='6px', flex_wrap='wrap'))
right   = widgets.VBox([info_html, btn_row, detail_img])

# ── map figure ────────────────────────────────────────────────────────────────
fig_map, ax_map = plt.subplots(figsize=(6, 6))
fig_map.patch.set_facecolor('#111111')
ax_map.set_facecolor('#111111')
for sp in ax_map.spines.values(): sp.set_edgecolor('#444')
ax_map.tick_params(colors='#777', labelsize=7)
fig_map.tight_layout(pad=0.5)
_sel_marker, = ax_map.plot([], [], 'w*', markersize=16, zorder=6,
                            markeredgecolor='#ff3333', markeredgewidth=1.2)


def _draw_tile(tilecode):
    _scatter_rows.clear()
    _selected_idx[0] = None
    _current_tile[0] = tilecode
    ax_map.cla()
    ax_map.set_facecolor('#111111')
    for sp in ax_map.spines.values(): sp.set_edgecolor('#444')
    ax_map.tick_params(colors='#777', labelsize=7)

    rgb, extent = _get_bg(tilecode)
    ax_map.imshow(rgb, origin='lower', extent=extent,
                  interpolation='nearest', aspect='equal')
    ax_map.set_xlim(extent[0], extent[1])
    ax_map.set_ylim(extent[2], extent[3])
    ax_map.set_title(tilecode, color='white', fontsize=9)
    ax_map.set_xlabel('X (m RD)', color='#777', fontsize=7)
    ax_map.set_ylabel('Y (m RD)', color='#777', fontsize=7)

    tile_inv = inv[inv['tilecode'] == tilecode]
    for lbl_code, grp in tile_inv.groupby('label'):
        name = LABEL_NAMES.get(lbl_code, f'Label {lbl_code}')
        fc   = [_dot_style(r)[0] for _, r in grp.iterrows()]
        ec   = [_dot_style(r)[1] for _, r in grp.iterrows()]
        ew   = [_dot_style(r)[2] for _, r in grp.iterrows()]
        al   = [_dot_style(r)[3] for _, r in grp.iterrows()]
        sc   = ax_map.scatter(
            grp['centroid_x'].values, grp['centroid_y'].values,
            c=fc, s=90, zorder=5, label=name,
            edgecolors=ec, linewidths=ew, alpha=None,
            picker=True, pickradius=8,
        )
        _scatter_rows.append((sc, grp.reset_index()))

    ax_map.legend(facecolor='#1e1e1e', labelcolor='white',
                  edgecolor='#444', fontsize=7, loc='upper right')
    progress_html.value = _progress_str(tilecode)
    fig_map.canvas.draw_idle()


def _select_cluster(row):
    _selected_idx[0] = row.name if hasattr(row, 'name') else row['index']
    _sel_marker.set_data([row['centroid_x']], [row['centroid_y']])
    fig_map.canvas.draw_idle()

    lbl   = int(row['label'])
    flbl  = row.get('final_label', None)
    fname = LABEL_NAMES.get(int(flbl), f'{flbl}') if pd.notna(flbl) else '—'
    nr    = row.get('needs_review', False)
    info_html.value = (
        f'<span style="color:#ccc;font-size:12px">'
        f'<b>#{int(row["cluster_idx"])}  '
        f'{LABEL_NAMES.get(lbl, str(lbl))}</b> · '
        f'{row.get("label_source","")} · '
        f'{int(row["n_raw_pts"]):,} pts · {float(row["area_m2"]):.2f} m²'
        f'<br>confirmed: <b>{fname}</b>'
        + ('  <span style="color:#ff6666">⚑ flagged</span>' if nr else '') +
        f'</span>'
    )
    # pre-select the current auto-label in the dropdown
    target = int(flbl) if pd.notna(flbl) else lbl
    label_dd.value = target if target in dict(LABEL_OPTIONS).values() else 0
    detail_img.value = render_detail(row)


# ── pick event ────────────────────────────────────────────────────────────────
def _on_pick(event):
    for sc, grp in _scatter_rows:
        if event.artist is sc:
            row = grp.iloc[event.ind[0]]
            _select_cluster(row)
            break

fig_map.canvas.mpl_connect('pick_event', _on_pick)


# ── labeling buttons ──────────────────────────────────────────────────────────
def _current_row():
    idx = _selected_idx[0]
    if idx is None:
        return None
    return inv.loc[idx] if idx in inv.index else None

def _apply_label(final_lbl, source, flag=False):
    row = _current_row()
    if row is None:
        info_html.value = '<span style="color:#cc4444">No cluster selected</span>'
        return
    idx = _selected_idx[0]
    inv.at[idx, 'final_label']        = final_lbl
    inv.at[idx, 'label_source_final'] = source
    inv.at[idx, 'needs_review']       = flag
    _save_inv()
    _draw_tile(_current_tile[0])        # redraw dots to reflect new status
    _select_cluster(inv.loc[idx])       # refresh panel

def _on_accept(_):
    row = _current_row()
    if row is not None:
        _apply_label(int(row['label']), 'confirmed')

def _on_update(_):
    _apply_label(label_dd.value, 'manual')

def _on_flag(_):
    row = _current_row()
    if row is not None:
        lbl = int(row.get('final_label', row['label']))
        _apply_label(lbl, 'manual', flag=True)

btn_accept.on_click(_on_accept)
btn_update.on_click(_on_update)
btn_flag.on_click(_on_flag)


# ── tile dropdown ─────────────────────────────────────────────────────────────
tile_dd = widgets.Dropdown(
    options=TILECODES, value=TILECODES[0],
    description='Tile:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '40px'},
)

def _on_tile(change):
    if change['name'] == 'value':
        detail_img.value = b''
        info_html.value  = '<span style="color:#666;font-size:12px">loading…</span>'
        _draw_tile(change['new'])
        info_html.value  = '<span style="color:#666;font-size:12px">click a cluster dot</span>'

tile_dd.observe(_on_tile)

# ── layout ────────────────────────────────────────────────────────────────────
header = widgets.HBox([tile_dd, progress_html],
                      layout=widgets.Layout(gap='20px', align_items='center'))
display(widgets.VBox([
    header,
    widgets.HBox([fig_map.canvas, right],
                 layout=widgets.Layout(gap='20px', align_items='flex-start')),
]))

_draw_tile(TILECODES[0])

### Background colour legend
| Colour | Label |
|---|---|
| 🔴 Dark red | Road |
| 🔵 Blue | Building |
| ⚫ Mid-grey | Ground / unknown |
| 🟢 Green | Tree |
| 🟠 Orange | Car |
| 🟡 Yellow | Street light |

### Dot border meaning
| Border | Meaning |
|---|---|
| Thin dark | Not yet reviewed |
| White | Confirmed |
| Red | Flagged for review |